# SNCP-PPO Social Navigation — Colab Notebook

End-to-end notebook for training and evaluating an **LTC + PPO** crowd-aware navigation policy.

## Run order
1. **Setup** — clone repo, install deps, optional Drive mount
2. **Smoke test** — verify env + model + a tiny training loop
3. **Train** — vectorized curriculum training (preflight gate → full run)
4. **Evaluate** — regime-matched density sweep + gates
5. **Visualize** — trajectory plot + GIFs
6. **Analyze** — learning curves from the training CSV
7. **Persist** — download checkpoint + evidence bundle

## Current run: v22 — paper regime + the paper's learning rate (1e-4)

v21 reproduced the paper's regime (robot **1.0 m/s** + parity ORCA pedestrians + goal noise + `max_time 15`) but fell short (best holdout min 38% vs the paper's 93–95%). Local attribution probes isolated the cause: at equal budget, **LR 5e-5 → 1e-4** (the paper's Table 1 value) took fixed-N=5 success from ~4% to 42%. Our 5e-5 dated from v11 ("stability") and starved learning. **v22 = v21 config with `LR = 1e-4`, single variable.** A pre-MLP (Eq 11) probe added nothing, so it stays off.

- **v18** remains the real-robot (0.26 m/s) baseline — best generalist min 70%.
- Architecture: SNCPPolicy = 3 NCP/LTC encoders (robot/temporal/spatial) + attention + actor-critic.
- Obs (robot-local): `robot_node` (7), `spatial_edges` (H×6 = pos + rel-vel + goal-dir), `temporal_edges` (2).
- Pedestrians: pure-Python **ORCA** (invisible robot — they avoid each other, not the robot).
- Best checkpoint = `min(success across holdout scenarios)` (rewards generalists).

## Colab tips
- **Runtime → Change runtime type → A100** recommended (~4–5 h for 2.5M steps). T4/L4 work but ~2–3× slower.
- Mount Drive (Section 1.4) so checkpoints/logs survive a disconnect.
- Pedestrian sim (ORCA) is CPU-side, so more vCPUs also help throughput.

## 1. Setup

### 1.1 GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1.2 Clone / update repository

Re-run after any push to pull the latest code. Note: this updates the repo files on the VM, **not** an already-open notebook — reopen the notebook from GitHub to get notebook changes.

In [ ]:
import os
REPO_URL = 'https://github.com/heimdilon/sncp-ppo-crowdnav.git'
REPO_DIR = '/content/sncp-ppo-crowdnav'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull --rebase

%cd {REPO_DIR}
!git log --oneline -1

### 1.3 Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import torch
print(f'torch     {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'          device: {torch.cuda.get_device_name(0)}')

import gymnasium, ncps, numpy, matplotlib
print(f'gymnasium {gymnasium.__version__}')
print(f'ncps      {ncps.__version__}')
print(f'numpy     {numpy.__version__}')

### 1.4 (Optional) Mount Google Drive

Set `USE_DRIVE = True` to persist `checkpoints/` and `logs/` across sessions (recommended for long runs — a disconnect mid-training otherwise loses everything).

In [ ]:
USE_DRIVE = False  # set True to persist runs across Colab sessions
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sncp-ppo-crowdnav-runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_PROJECT_DIR}/logs', exist_ok=True)
    import shutil
    for sub in ('checkpoints', 'logs'):
        local = f'{REPO_DIR}/{sub}'
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            for f in os.listdir(local):
                dst = f'{DRIVE_PROJECT_DIR}/{sub}/{f}'
                if not os.path.exists(dst):
                    shutil.copy2(f'{local}/{f}', dst)
            shutil.rmtree(local)
        os.symlink(f'{DRIVE_PROJECT_DIR}/{sub}', local)
    print(f'Drive-backed dirs: {DRIVE_PROJECT_DIR}/{{checkpoints,logs}}')
else:
    print('Drive mount skipped (USE_DRIVE=False). Files are lost when the Colab session ends.')

## 2. Smoke tests

Fast sanity checks before spending GPU hours. Env + model first, then a 50-episode single-env training loop (legacy path) that exercises curriculum, holdout, value clipping, LR schedule, and the per-update diagnostics line.

In [ ]:
!python test_env.py

In [ ]:
!python test_model.py

In [ ]:
# 50-episode single-env smoke. NOT the full run (that's Section 3). Replay is 0
# here so this stays a quick baseline check of the single-env path.
import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--episodes', '50',
    '--num_humans', '5',
    '--seed', '42',
    '--eval_freq', '25',
    '--holdout_episodes', '3',
    '--holdout_scenarios', 'easy', 'hard',
    '--update_freq', '5',
    '--log_freq', '10',
    '--curriculum_replay_ratio', '0.0',
    '--save_path', 'checkpoints/sncp_ppo_smoke.pt',
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')

## 3. Full training (v22 — paper regime + paper LR 1e-4)

Single variable vs v21: `LR = 5e-5 → 1e-4`. Probe evidence (300k budget, everything else equal): easy-warmup success 0%→72%, fixed-N=5 success 4%→42% (still rising), timeout 45%→2%.

Run the **preflight gate first** — it fails fast if the notebook config drifts from the verified v22 spec, so you don't waste GPU time on a stale setup.

| Argument | v22 value | note |
| --- | --- | --- |
| `--lr` | **1e-4** | the v22 variable (paper Table 1; v21 used 5e-5) |
| `--robot_vpref` / `--human_vpref_override` | 1.0 / 1.0 | parity, paper regime |
| `--human_goal_noise` | 2.0 | spread crossings (no center funnel) |
| `--max_time` | 15.0 | fast robot reaches ~8 m in ~8 s |
| `--curriculum_replay_ratio` | 0.20 | anti-forgetting replay |
| `--comfort_coeff` | 6.0 | comfort penalty -6·I_sp |

### 3.1 Preflight gate

In [ ]:
# Fail-fast check of the v22 notebook config, eval wiring, and the v21 baseline.
import subprocess, sys
cmd = [sys.executable, 'verify_v16_run_ready.py', '--output', 'eval_v22/run_readiness.md']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

from IPython.display import Markdown, display
with open('eval_v22/run_readiness.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

### 3.2 Full training run

In [ ]:
# v22 = v21 paper regime + the paper's learning rate (1e-4). Single variable.
# Probe-validated: at equal budget, 5e-5 -> 1e-4 took fixed-N=5 success 4% -> 42%.
# Pre-MLP (Eq 11) probe added nothing, so that flag stays OFF (kept for v23).
# v18 remains the 0.26 m/s real-robot baseline.
NUM_ENVS = 16
HORIZON = 128
TOTAL_STEPS = 2_500_000
SEED = 42
LR = 1e-4              # THE v22 variable (paper Table 1; v21 ran 5e-5)
TARGET_KL = 0.01
REPLAY_RATIO = 0.20
COMFORT_COEFF = 6.0
MAX_TIME = 15.0        # fast robot reaches ~8m in ~8s (paper uses 12.5s)
ROBOT_VPREF = 1.0      # paper's robot speed (vs TurtleBot 0.26)
HUMAN_VPREF = 1.0      # ORCA pedestrians at parity (paper regime)
HUMAN_GOAL_NOISE = 2.0  # spread pedestrian goals so they don't funnel through center
SAVE_PATH = 'checkpoints/sncp_ppo_v22.pt'

import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--num_envs', str(NUM_ENVS),
    '--horizon', str(HORIZON),
    '--total_steps', str(TOTAL_STEPS),
    '--eval_freq_updates', '20',
    '--num_humans', '10',
    '--seed', str(SEED),
    '--lr', str(LR),
    '--lr_end_factor', '0.1',
    '--target_kl', str(TARGET_KL),
    '--curriculum_replay_ratio', str(REPLAY_RATIO),
    '--comfort_coeff', str(COMFORT_COEFF),
    '--max_time', str(MAX_TIME),
    '--robot_vpref', str(ROBOT_VPREF),
    '--human_vpref_override', str(HUMAN_VPREF),
    '--human_goal_noise', str(HUMAN_GOAL_NOISE),
    '--holdout_scenarios', 'easy', 'hard', 'circle',
    '--holdout_episodes', '50',
    '--save_path', SAVE_PATH,
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')
if p.returncode != 0:
    raise SystemExit(p.returncode)

### Resuming after a disconnect

There's no resume CLI. With `USE_DRIVE=True` the best checkpoint is safe in Drive; the simplest restart is to rerun 3.2 (optionally with a different `--seed`). The best-checkpoint logic keeps the highest-`min(success)` weights regardless of later collapse.

## 4. Evaluation (regime-matched)

The eval **must** use the same regime the checkpoint trained in (robot 1.0 + parity peds + `max_time 15`), so the density sweep passes those flags. **v21 lesson:** the old gates were regime-invalid at 1.0 m/s (the 0.26-regime beeline threshold auto-failed every run), so v22 compares against the regime-matched `eval_v21/density_sweep.json` baseline (same physics — the LR ablation) with the no-beeline gate scaled to 1.0 m/s (baseline 32 steps, margin 8). Exits nonzero only on a hard artifact-verifier fail.

In [ ]:
CHECKPOINT = 'checkpoints/sncp_ppo_v22.pt'  # v22 = v21 regime + paper LR 1e-4
EVAL_OUT = 'eval_v22'
EVAL_SEED = 100      # different from training seeds for a fair eval
EVAL_EPISODES = 50

import subprocess, sys
cmd = [
    sys.executable, 'run_post_eval.py',
    '--version', '22',
    '--densities', '1', '3', '5', '8', '10',
    '--scenario', 'hard',
    '--n_episodes', str(EVAL_EPISODES),
    '--seed', str(EVAL_SEED),
    '--trajectory_densities', '5', '10',
    '--robot_vpref', '1.0',
    '--human_vpref_override', '1.0',
    '--human_goal_noise', '2.0',
    '--max_time', '15.0',
    '--baseline_json', 'eval_v21/density_sweep.json',
    '--baseline_nav_steps', '32',
    '--nav_margin_steps', '8',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

from IPython.display import Image, Markdown, display
for name in ['artifact_verification.md', 'comparison_vs_v15.md', 'training_diagnostics.md', 'report.md']:
    with open(f'{EVAL_OUT}/{name}', 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
display(Image(f'{EVAL_OUT}/density_sweep.png'))

## 5. Visualize trajectories

All visualizers run in the **v22 regime** (robot 1.0 + parity peds + goal noise + max_time 15) so the plots reflect what the checkpoint actually learned.

In [ ]:
# Single trajectory plot — first successful episode out of 20 tries.
!python visualize_trajectory.py \
    --checkpoint {CHECKPOINT} \
    --output trajectory_plot.png \
    --num_humans 10 \
    --scenario hard \
    --robot_vpref 1.0 --human_vpref_override 1.0 --human_goal_noise 2.0 --max_time 15.0 \
    --seed 42

from IPython.display import Image, display
display(Image('trajectory_plot.png'))

In [ ]:
# Animated GIF for a single scenario.
!python visualize_trajectory_gif.py --checkpoint {CHECKPOINT} --num_humans 10 --scenario hard

from IPython.display import Image, display
import glob
gifs = sorted(glob.glob('*.gif'))
if gifs:
    print(f'Generated: {gifs}')
    display(Image(gifs[-1]))

## 6. Training curves

Plots the newest training CSV and shows the v22 diagnostics + artifact-verification reports.

In [ ]:
import glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if not csv_files:
    print('No training CSVs found. Run Section 3 first.')
else:
    latest_csv = csv_files[-1]
    print(f'Plotting: {latest_csv}')
    !python plot_training.py --csv {latest_csv} --output training_curves_colab.png --window 50
    from IPython.display import Image, Markdown, display
    display(Image('training_curves_colab.png'))
    for path in ['eval_v22/training_diagnostics.md', 'eval_v22/artifact_verification.md']:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                display(Markdown(f.read()))
        except FileNotFoundError:
            print(f'{path} not found. Run the Section 4 evaluation cell first.')

### Inspect CSV in pandas (optional)

In [ ]:
import pandas as pd, glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f'Rows: {len(df)}')
    print(f'Columns: {list(df.columns)}')
    print('\nPhase distribution:')
    print(df['scenario'].value_counts().sort_index())
    print('\nHoldout success per eval (event points):')
    holdout_cols = [c for c in df.columns if c.startswith('holdout_') and c.endswith('_success')]
    if holdout_cols:
        hdf = df[holdout_cols].drop_duplicates()
        hdf.index = df.loc[hdf.index, 'episode']
        print(hdf.tail(10))

## 7. Persist results

With `USE_DRIVE=True` the checkpoint + logs are already in Drive. Otherwise set `DOWNLOAD = True` to grab the checkpoint, latest CSV + curve, and the full `eval_v22` evidence bundle before the session ends.

In [ ]:
from google.colab import files
import glob, os, shutil

DOWNLOAD = False  # set True to trigger browser download dialogs
if DOWNLOAD:
    if os.path.exists(SAVE_PATH):
        files.download(SAVE_PATH)
    for pattern in ['logs/training_*.csv', 'training_curves_colab.png']:
        for f in sorted(glob.glob(pattern))[-1:]:
            files.download(f)
    if os.path.isdir('eval_v22'):
        archive = shutil.make_archive('eval_v22_artifacts', 'zip', 'eval_v22')
        files.download(archive)

## 8. Notes & roadmap (current: v22 — LR fix in the paper regime)

`README.md` is stale; `AGENTS.md` + the code are the source of truth.

### Story so far
- **v18** (paper-faithful `r_g`): breakthrough; fixed timeout; best generalist min 56→70%. The real-robot (0.26 m/s) baseline.
- **v19** (clamp `I_sp`): negative — the remaining high-N collision is not a reward issue.
- **v20** (ORCA pedestrians): paper's CrowdSim regime; GIF inspection refuted the "SFM knots" idea.
- **v21** (paper regime: robot 1.0 + parity + max_time 15): NEGATIVE at LR 5e-5 — best holdout min 38%, collision-dominant at high N.
- **v22** (this run): v21 regime + **LR 1e-4** (probe-validated single variable). Eval gates are regime-matched (v21 baseline; beeline scaled to 1.0 m/s).

### After v22
- **v22 ≈ paper (90%+):** implementation validated; write up with v18 as the hardware track.
- **v22 improves but plateaus:** next single variables = pre-MLP (`--pre_mlp`, coded + tested), IL warm-start (BC from an ORCA-robot expert), longer budget.
- **v22 ≈ v21:** LR wasn't the binding constraint at full scale; re-attribute with longer probes.

### Evaluation gates
After Section 3.2 produces `checkpoints/sncp_ppo_v22.pt`, run Section 4 (same 1.0/parity/max_time-15 regime) and read `eval_v22/artifact_verification.md`, then check success at N=5/8/10 vs the paper's 93–95%.